# TabICL vs TabPFN vs CatBoost

Focused three-way comparison on the small tabular benchmark.
- **TabICL** — tabular foundation model (in-context learning, zero HP tuning beyond ensemble size and softmax temperature)
- **TabPFN** — earlier foundation model (10-class cap, slower than TabICL on the same data)
- **CatBoost** — tuned via Optuna (50 trials per outer fold), strongest gradient booster on this benchmark

In [1]:
import joblib
from collections import Counter
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(context='poster')

In [2]:
from benchmark.checkpoints import load_by_dataset

ckpt = load_by_dataset()
db = pd.read_json('database.json').T
db['nrow'] = np.minimum(10000, db['nrow'])

MODELS = {
    'tabicl':   'TabICL',
    'tabpfn3':  'TabPFN-3',
    'catboost': 'CatBoost',
}
FOLD_COLS  = ['prauc_fold_1', 'prauc_fold_2', 'prauc_fold_3', 'prauc_fold_4']
SPLIT_COLS = ['prauc_split_1', 'prauc_split_2', 'prauc_split_3', 'prauc_split_4']

rows = []
for ds, v in ckpt.items():
    for key, label in MODELS.items():
        if key in v:
            scores = v[key]['scores']
            if not any(np.isnan(s) for s in scores):
                rows.append({'dataset': ds, 'model': label,
                              **dict(zip(FOLD_COLS, scores)),
                              'time': v[key]['time']})
df = pd.DataFrame(rows)
df.columns = ['dataset', 'model'] + SPLIT_COLS + ['time']
df['mean_prauc'] = df[SPLIT_COLS].mean(axis=1)
for col in db.columns:
    df[col] = df['dataset'].map(db[col]).to_numpy()

# Only keep datasets where all 3 models have valid scores (head-to-head)
counts = df.groupby('dataset')['model'].nunique()
shared = counts[counts == len(MODELS)].index
df = df[df['dataset'].isin(shared)].reset_index(drop=True)
print(f'Shared datasets (all 3 models valid): {len(shared)}')

# Drop trivially-easy datasets where the worst model still > 0.99
df = df.groupby('dataset').filter(lambda x: x['mean_prauc'].min() < 0.99).reset_index(drop=True)
active = set(df['dataset'])
print(f'After dropping trivially-easy: {len(active)}')

Shared datasets (all 3 models valid): 142
After dropping trivially-easy: 111


In [3]:
summary = df.groupby('model').agg(
    n=('dataset', 'count'),
    mean_prauc=('mean_prauc', 'mean'),
    median_prauc=('mean_prauc', 'median'),
    median_time_s=('time', 'median'),
    mean_time_s=('time', 'mean'),
    total_time_h=('time', lambda x: x.sum() / 3600),
).round(4).sort_values('mean_prauc', ascending=False)
print(summary.to_string())

            n  mean_prauc  median_prauc  median_time_s  mean_time_s  total_time_h
model                                                                            
TabICL    111      0.8180        0.9173       288.5532     811.0392       25.0070
TabPFN-3  111      0.8158        0.9132      1024.1983    1569.5096       48.3932
CatBoost  111      0.7898        0.8831       257.3238    1652.8218       50.9620


## Pairwise win counts

How often each model beats each other on the same dataset (mean PR-AUC across 4 folds).

In [4]:
pivot = df.pivot_table(index='dataset', columns='model', values='mean_prauc')
labels = list(MODELS.values())

win = pd.DataFrame(index=labels, columns=labels, dtype=object)
for a in labels:
    for b in labels:
        if a == b:
            win.loc[a, b] = '—'
        else:
            wins = (pivot[a] > pivot[b]).sum()
            total = pivot[[a, b]].dropna().shape[0]
            win.loc[a, b] = f'{wins}/{total} ({wins/total*100:.0f}%)'
print('Wins (row beats column):')
print(win.to_string())
print()
for a in labels:
    for b in labels:
        if a >= b:
            continue
        sub = pivot[[a, b]].dropna()
        delta = sub[a] - sub[b]
        print(f'{a} − {b}:  mean Δ = {delta.mean():+.4f}  median Δ = {delta.median():+.4f}  '
              f'(margin ≥0.01: {a} wins {(delta>=0.01).sum()}, {b} wins {(delta<=-0.01).sum()}, ties {((-0.01<delta)&(delta<0.01)).sum()})')

Wins (row beats column):
                TabICL      TabPFN-3      CatBoost
TabICL               —  60/111 (54%)  96/111 (86%)
TabPFN-3  51/111 (46%)             —  90/111 (81%)
CatBoost  15/111 (14%)  21/111 (19%)             —

TabICL − TabPFN-3:  mean Δ = +0.0022  median Δ = +0.0001  (margin ≥0.01: TabICL wins 19, TabPFN-3 wins 6, ties 86)
CatBoost − TabICL:  mean Δ = -0.0281  median Δ = -0.0122  (margin ≥0.01: CatBoost wins 4, TabICL wins 60, ties 47)
CatBoost − TabPFN-3:  mean Δ = -0.0260  median Δ = -0.0107  (margin ≥0.01: CatBoost wins 6, TabPFN-3 wins 57, ties 48)


In [5]:
# Per-dataset delta vs CatBoost (baseline for this comparison)
cb = pivot['CatBoost']
delta_df = pd.DataFrame({
    'TabICL − CatBoost': pivot['TabICL'] - cb,
    'TabPFN − CatBoost': pivot['TabPFN-3'] - cb,
}).reset_index().melt(id_vars='dataset', var_name='comparison', value_name='delta')

fig, ax = plt.subplots(figsize=(14, 6))
sns.boxenplot(data=delta_df, x='delta', y='comparison', ax=ax)
ax.axvline(0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Mean PR-AUC − CatBoost (positive = better than tuned CatBoost)')
ax.set_title(f'Foundation models vs tuned CatBoost ({len(active)} datasets)')
plt.tight_layout()
plt.savefig('results/tabicl_vs_others_delta_boxen.png', dpi=120)
plt.show()

/var/folders/jj/h3091n1n5jbcjnyy2cd98h140000gn/T/ipykernel_7761/3974212871.py:15: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [6]:
fig, axes = plt.subplots(1, 3, figsize=(20, 6.5))
pairs = [
    ('TabICL', 'CatBoost'),
    ('TabICL', 'TabPFN-3'),
    ('TabPFN-3', 'CatBoost'),
]
for ax, (a, b) in zip(axes, pairs):
    sub = pivot[[a, b]].dropna()
    ax.scatter(sub[b], sub[a], alpha=0.7, s=50)
    lims = [min(sub.min()), max(sub.max())]
    ax.plot(lims, lims, 'k--', linewidth=1)
    ax.set_xlabel(b, fontsize=14)
    ax.set_ylabel(a, fontsize=14)
    n_win = (sub[a] > sub[b]).sum()
    ax.set_title(f'{a} wins {n_win}/{len(sub)}', fontsize=14)
fig.suptitle('Mean PR-AUC per dataset', fontsize=16)
plt.tight_layout()
plt.savefig('results/tabicl_vs_others_scatter.png', dpi=120)
plt.show()

/var/folders/jj/h3091n1n5jbcjnyy2cd98h140000gn/T/ipykernel_7761/2656458244.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [7]:
rank = pivot.rank(axis=1, ascending=False, method='average')
rank_counts = pd.DataFrame(index=labels)
for r in range(1, len(labels) + 1):
    rank_counts[f'rank_{r}'] = (rank == r).sum()
rank_counts['mean_rank'] = rank.mean()
print('Rank distribution (lower mean_rank = better):')
print(rank_counts.sort_values('mean_rank').to_string())

Rank distribution (lower mean_rank = better):
          rank_1  rank_2  rank_3  mean_rank
TabICL        53      50       8   1.594595
TabPFN-3      46      49      16   1.729730
CatBoost      12      12      87   2.675676


In [8]:
# Time per dataset comparison
time_pivot = df.pivot_table(index='dataset', columns='model', values='time')
ratio = pd.DataFrame({
    'TabICL / CatBoost': time_pivot['TabICL'] / time_pivot['CatBoost'],
    'TabPFN-3 / CatBoost': time_pivot['TabPFN-3'] / time_pivot['CatBoost'],
    'TabICL / TabPFN-3':   time_pivot['TabICL'] / time_pivot['TabPFN-3'],
})
print('Time ratio (X / Y means X is N× slower than Y):')
print(ratio.agg(['median', 'mean']).round(2).to_string())

Time ratio (X / Y means X is N× slower than Y):
        TabICL / CatBoost  TabPFN-3 / CatBoost  TabICL / TabPFN-3
median               0.97                 4.54               0.26
mean                 2.60                 9.39               0.43


In [9]:
# Worth it? PR-AUC gain over CatBoost vs time cost
fig, ax = plt.subplots(figsize=(14, 8))
for name, color in [('TabICL', 'tab:blue'), ('TabPFN-3', 'tab:orange')]:
    delta = (pivot[name] - cb)
    t_ratio = (time_pivot[name] / time_pivot['CatBoost'])
    valid = (~delta.isna()) & (~t_ratio.isna())
    ax.scatter(t_ratio[valid], delta[valid], alpha=0.65, s=60, color=color, label=name)
ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.set_xscale('log')
ax.set_xlabel('Train time / CatBoost (log scale)')
ax.set_ylabel('Mean PR-AUC − CatBoost')
ax.set_title(f'Accuracy gain vs compute cost ({len(active)} datasets)')
ax.legend()
plt.tight_layout()
plt.savefig('results/tabicl_vs_others_cost_vs_gain.png', dpi=120)
plt.show()

/var/folders/jj/h3091n1n5jbcjnyy2cd98h140000gn/T/ipykernel_7761/1930254608.py:16: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [10]:
delta_vs_cb = pd.DataFrame({
    'TabICL': pivot['TabICL'] - cb,
    'TabPFN-3': pivot['TabPFN-3'] - cb,
    'CatBoost': cb,
}).dropna()
print('Top 10 datasets where TabICL beats CatBoost most:')
print(delta_vs_cb.sort_values('TabICL', ascending=False).head(10).to_string())
print()
print('Top 10 datasets where CatBoost beats TabICL most:')
print(delta_vs_cb.sort_values('TabICL').head(10).to_string())

Top 10 datasets where TabICL beats CatBoost most:
                                 TabICL  TabPFN-3  CatBoost
dataset                                                    
hill-valley-with-noise         0.440654  0.440820  0.556032
hill-valley-without-noise      0.380000  0.379813  0.619862
plant-species-leaves-shape     0.167679  0.166580  0.732072
meta-data                      0.145879  0.069188  0.057176
statlog-vehicle-silhouettes    0.118532  0.105862  0.838694
autoUniv-au7-cpd1-500          0.093336  0.075684  0.473692
teaching-assistant-evaluation  0.084400  0.093536  0.673535
indian-liver-patient           0.075713  0.059895  0.501778
robot-failure-lp5              0.072913  0.064989  0.745027
movement-libras                0.069708  0.065806  0.903615

Top 10 datasets where CatBoost beats TabICL most:
                          TabICL  TabPFN-3  CatBoost
dataset                                             
planning-relax         -0.050895 -0.017613  0.363742
monks3              

In [11]:
# Are TabICL and TabPFN-3 blind on the same datasets?
icl_d = pivot['TabICL'] - cb
pfn_d = pivot['TabPFN-3'] - cb
joint = pd.concat([icl_d.rename('TabICL'), pfn_d.rename('TabPFN-3')], axis=1).dropna()
corr = joint.corr().iloc[0, 1]
both_win  = ((joint['TabICL'] >= 0.01) & (joint['TabPFN-3'] >= 0.01)).sum()
both_lose = ((joint['TabICL'] <= -0.01) & (joint['TabPFN-3'] <= -0.01)).sum()
disagree  = ((np.sign(joint['TabICL']) != np.sign(joint['TabPFN-3'])) &
              (joint.abs() >= 0.01).any(axis=1)).sum()
print(f'corr(TabICL-CatBoost, TabPFN-CatBoost) over {len(joint)} datasets: {corr:.3f}')
print(f'both foundation models beat CatBoost (Δ ≥ +0.01): {both_win}')
print(f'both foundation models lose to CatBoost (Δ ≤ −0.01): {both_lose}')
print(f'disagree (sign-flip with |Δ| ≥ 0.01 somewhere): {disagree}')

fig, ax = plt.subplots(figsize=(9, 9))
ax.scatter(joint['TabPFN-3'], joint['TabICL'], alpha=0.65, s=50)
ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.axvline(0, color='black', linestyle='--', linewidth=1)
lim = max(joint.abs().max()) * 1.05
ax.plot([-lim, lim], [-lim, lim], 'k:', linewidth=1, alpha=0.5)
ax.set_xlim(-lim, lim); ax.set_ylim(-lim, lim)
ax.set_xlabel('TabPFN − CatBoost')
ax.set_ylabel('TabICL − CatBoost')
ax.set_title(f'Foundation models share blind spots (corr={corr:.2f})')
plt.tight_layout()
plt.savefig('results/tabicl_vs_others_foundation_corr.png', dpi=120)
plt.show()

corr(TabICL-CatBoost, TabPFN-CatBoost) over 111 datasets: 0.981
both foundation models beat CatBoost (Δ ≥ +0.01): 55
both foundation models lose to CatBoost (Δ ≤ −0.01): 4
disagree (sign-flip with |Δ| ≥ 0.01 somewhere): 3


/var/folders/jj/h3091n1n5jbcjnyy2cd98h140000gn/T/ipykernel_7761/120825802.py:27: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [12]:
# Does TabICL's advantage over CatBoost depend on dataset size?
nrow = df.drop_duplicates('dataset').set_index('dataset')['nrow']
size_df = pd.DataFrame({
    'nrow':  nrow,
    'delta': pivot['TabICL'] - cb,
}).dropna()
bins = [0, 200, 500, 1000, 3000, 10000]
size_df['bucket'] = pd.cut(size_df['nrow'], bins=bins, include_lowest=True)
agg = size_df.groupby('bucket', observed=True)['delta'].agg(['count', 'mean', 'median'])
agg['win_rate'] = size_df.groupby('bucket', observed=True)['delta'].apply(lambda x: (x > 0).mean())
print('TabICL − CatBoost delta by dataset size:')
print(agg.round(4).to_string())

fig, ax = plt.subplots(figsize=(12, 6))
sns.boxenplot(data=size_df, x='bucket', y='delta', ax=ax)
ax.axhline(0, color='black', linestyle='--', linewidth=1)
ax.set_xlabel('Number of training samples (bucketed)')
ax.set_ylabel('TabICL − CatBoost')
ax.set_title(f'TabICL advantage vs dataset size ({len(size_df)} datasets)')
plt.tight_layout()
plt.savefig('results/tabicl_vs_catboost_by_size.png', dpi=120)
plt.show()

TabICL − CatBoost delta by dataset size:
                   count    mean  median  win_rate
bucket                                            
(-0.001, 200.0]       14  0.0234  0.0184    0.8571
(200.0, 500.0]        20  0.0214  0.0137    0.9500
(500.0, 1000.0]       17  0.0320  0.0153    0.8824
(1000.0, 3000.0]      26  0.0523  0.0088    0.9231
(3000.0, 10000.0]     34  0.0137  0.0081    0.7647


/var/folders/jj/h3091n1n5jbcjnyy2cd98h140000gn/T/ipykernel_7761/3244228817.py:22: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Conclusions

110 shared non-trivial datasets, TabICL vs TabPFN-3 vs tuned CatBoost.

### The two foundation models are interchangeable here

TabICL 0.8172, TabPFN-3 0.8150. TabICL wins 59/110 (54%), mean delta +0.0022,
median +0.0001. At a 0.01 margin, 85 of 110 datasets are ties. Choose between them
on cost and coverage, not on scores.

### Both beat tuned CatBoost consistently

TabICL wins 95/110 (86%), TabPFN-3 89/110 (81%). Mean delta +0.027 and +0.025,
median +0.012 and +0.011. CatBoost beats both by more than 0.01 on 4 datasets.

The win rate holds across dataset size — 86%, 95%, 88%, 92%, 76% from the smallest
bucket to the largest — so this is not a small-data effect.

### Cost

Median per dataset: TabICL 287 s, CatBoost 257 s, TabPFN-3 1015 s. At the median
TabICL costs the same as 50-trial-Optuna CatBoost (0.99x) and TabPFN-3 costs 4.6x.
Means are higher (2.6x and 9.5x) because large datasets pull foundation-model time
up disproportionately.

### They fail together, but they rarely fail

The correlation between their deltas vs CatBoost is 0.980, so when one struggles the
other does too. But the direction matters: both beat CatBoost by more than 0.01 on
54 datasets and both lose by more than 0.01 on 4.

An earlier version of this notebook read that same correlation as evidence of a
shared blind spot on small imbalanced medical data, and recommended always
co-training CatBoost. Those datasets were mis-scored: `pr_auc_score` reads
`y_prob[:, 1]` on binary problems, and the classical and foundation paths were
encoding labels in opposite orders. `blood-transfusion-service`, then reported at
CatBoost +0.36, is now TabICL +0.012. See
[Findings_notes.md](Findings_notes.md#label-ordering-silently-changed-the-metric).

The four datasets where CatBoost genuinely leads are `planning-relax`, `monks3`,
`appendicitis` and `pima-indians-diabetes`, by 0.011 to 0.051. That is not a pattern
worth engineering around.

### Where the foundation models win big

Almost entirely `hill-valley-with-noise` (+0.44) and `hill-valley-without-noise`
(+0.38), where CatBoost scores 0.556 and 0.620 against ~0.99. Synthetic structured
noise that gradient boosting cannot regularise away. Outside those two, the largest
margin is +0.15.

### Recommendation

1. **TabICL on CPU** — same median cost as tuned CatBoost, wins 86% of the time.
2. **TabPFN-3 when the data is wide or many-class**, which TabICL skips.
3. **Neither is the benchmark's strongest model.** Both AutoML frameworks beat them
   given 300 s per fold. See [FoundationModels_notes.md](FoundationModels_notes.md#against-automl).
